In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd

# QA based on Agentic AI data
inputs = [
    "What is Agentic AI and how does it differ from traditional AI models?",
    "What are the key components of an Agentic AI system?",
    "What are the common applications and challenges of Agentic AI?",
]

outputs = [
    "Agentic AI refers to AI systems that can plan, reason, use tools, and take actions to achieve user-defined goals with limited supervision. Unlike traditional AI models that simply respond to a single prompt, agentic systems break complex tasks into smaller steps, decide what information is needed, interact with external tools such as search engines, databases, or APIs, and adapt based on intermediate results.",
    "The key components of Agentic AI include: Reasoning (deciding what to do next), Memory (retaining useful context across steps), Tools (calling APIs, calculators, databases, OCR systems, or web search), Planning (decomposing large objectives into smaller tasks), and Reflection (evaluating outputs and correcting mistakes).",
    "Common applications of Agentic AI include customer support, document assistants, research automation, coding assistants, workflow automation, and enterprise knowledge management. Challenges include hallucinations, tool failures, security risks, latency, and cost. Best practices include validating outputs, limiting permissions, logging actions, using human approval for high-risk operations, and grounding responses with trusted knowledge sources through RAG.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = r"E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\data\goldens.csv"
df.to_csv(csv_path, index=False)

print(f"✅ CSV saved to: {csv_path}")
print(f"📊 Created {len(df)} QA pairs")
print(df)

✅ CSV saved to: E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\data\goldens.csv
📊 Created 3 QA pairs
                                            question  \
0  What is Agentic AI and how does it differ from...   
1  What are the key components of an Agentic AI s...   
2  What are the common applications and challenge...   

                                              answer  
0  Agentic AI refers to AI systems that can plan,...  
1  The key components of Agentic AI include: Reas...  
2  Common applications of Agentic AI include cust...  


In [ ]:
from langsmith import Client

client = Client()
dataset_name = "Rag_App_dataset"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['66541cb8-1443-4473-8004-a23cff7e898d',
  'd0376261-e0a1-40e5-84f8-732490b035b3',
  '2d608d00-b72c-4fad-8939-029ac32b740d'],
 'count': 3,
 'as_of': '2026-07-23T18:40:20.255111958Z'}

In [9]:
import sys
sys.path.append(r"E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = r"E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\data\Agentic_AI_RAG_Training.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5,
    search_type: str = "mmr",
    fetch_k: int = 20,
    lambda_mult: float = 0.5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
        search_type: "mmr" or "similarity"
        fetch_k: Number of documents to fetch for MMR
        lambda_mult: Diversity parameter for MMR
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists using Path
        data_file = Path(data_path)  # FIXED: Path is now defined
        if not data_file.exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        retriever = ingestor.build_retriever(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k,
            search_type=search_type,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "faiss_index"),
            search_type=search_type,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}



e:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# Test the function with a sample question
test_input = {"question": "What is Agentic AI and how does it differ from traditional AI models?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"session": "session_20260724_010410_2e4835fc", "timestamp": "2026-07-23T19:34:10.546633Z", "level": "info", "event": "ChatIngestor initialized"}
{"original": "Agentic_AI_RAG_Training.txt", "saved": "data\\session_20260724_010410_2e4835fc\\agentic_ai_rag_training_c41f2ac5.txt", "timestamp": "2026-07-23T19:34:10.554634Z", "level": "info", "event": "File saved"}
{"total": 1, "timestamp": "2026-07-23T19:34:10.556634Z", "level": "info", "event": "Upload completed"}


{"count": 1, "failed": 0, "timestamp": "2026-07-23T19:34:11.064783Z", "level": "info", "event": "Documents loaded"}
{"documents": 1, "chunks": 3, "timestamp": "2026-07-23T19:34:11.083140Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-07-23T19:34:11.086142Z", "level": "info", "event": "Loading embedding model"}
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4205.37it/s]
{"documents": 3, "path": "faiss_index\\session_20260724_010410_2e4835fc\\faiss_index", "timestamp": "2026-07-23T19:34:19.192195Z", "level": "info", "event": "Created FAISS index"}
{"count": 3, "timestamp": "2026-07-23T19:34:19.265971Z", "level": "info", "event": "Added documents to FAISS"}
{"search_type": "mmr", "timestamp": "2026-07-23T19:34:19.268967Z", "level": "info", "event": "Retriever created"}
{"provider": "groq", "model": "llama-3.3-70b-versatile", "timestamp": "2026-07-23T19:34:19.273480Z", "level": "info

Question: What is Agentic AI and how does it differ from traditional AI models?

Answer: Agentic AI refers to AI systems that can plan, reason, use tools, and take actions to achieve user-defined goals with limited supervision. It differs from traditional AI models in several ways:

1. **Autonomy**: Agentic AI systems can break complex tasks into smaller steps, decide what information is needed, and interact with external tools, whereas traditional AI models simply respond to a single prompt.
2. **Decision-making**: Agentic AI systems can make decisions and take actions based on intermediate results, whereas traditional AI models only provide a response to a prompt.
3. **Tool usage**: Agentic AI systems can interact with external tools such as search engines, databases, or APIs, whereas traditional AI models do not have this capability.
4. **Adaptability**: Agentic AI systems can adapt based on intermediate results, whereas traditional AI models do not have this ability.

Overall, Agen

In [21]:
from langsmith.evaluation import evaluate
from langsmith.evaluation import StringEvaluator

In [22]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"session": "session_20260724_012102_1328cf45", "timestamp": "2026-07-23T19:51:02.226717Z", "level": "info", "event": "ChatIngestor initialized"}
{"original": "Agentic_AI_RAG_Training.txt", "saved": "data\\session_20260724_012102_1328cf45\\agentic_ai_rag_training_42e1d2c1.txt", "timestamp": "2026-07-23T19:51:02.241714Z", "level": "info", "event": "File saved"}
{"total": 1, "timestamp": "2026-07-23T19:51:02.244715Z", "level": "info", "event": "Upload completed"}


Testing all questions from the dataset:



{"count": 1, "failed": 0, "timestamp": "2026-07-23T19:51:02.921310Z", "level": "info", "event": "Documents loaded"}
{"documents": 1, "chunks": 3, "timestamp": "2026-07-23T19:51:02.963254Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-07-23T19:51:02.965259Z", "level": "info", "event": "Loading embedding model"}
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3654.38it/s]
{"documents": 3, "path": "faiss_index\\session_20260724_012102_1328cf45\\faiss_index", "timestamp": "2026-07-23T19:51:10.025923Z", "level": "info", "event": "Created FAISS index"}
{"count": 3, "timestamp": "2026-07-23T19:51:10.110934Z", "level": "info", "event": "Added documents to FAISS"}
{"search_type": "mmr", "timestamp": "2026-07-23T19:51:10.114923Z", "level": "info", "event": "Retriever created"}
{"provider": "groq", "model": "llama-3.3-70b-versatile", "timestamp": "2026-07-23T19:51:10.117923Z", "level": "info

Q1: What is Agentic AI and how does it differ from traditional AI models?
A1: Agentic AI refers to AI systems that can plan, reason, use tools, and take actions to achieve user-defined goals with limited supervision. It differs from traditional AI models in several ways:

1. **Multi-step problem-solving**: Agentic AI can break down complex tasks into smaller steps, decide what information is needed, and interact with external tools to achieve a goal. Traditional AI models, on the other hand, typically respond to a single prompt.
2. **Autonomy**: Agentic AI systems can operate with limited supervision, making decisions and taking actions to achieve a goal. Traditional AI models often require more explicit guidance and oversight.
3. **Integration with external tools**: Agentic AI systems can interact with external tools such as search engines, databases, or APIs to gather information and execute tasks. Traditional AI models typically do not have this capability.
4. **Adaptability**: Agen

{"count": 1, "failed": 0, "timestamp": "2026-07-23T19:51:24.237527Z", "level": "info", "event": "Documents loaded"}
{"documents": 1, "chunks": 3, "timestamp": "2026-07-23T19:51:24.255525Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-07-23T19:51:24.258518Z", "level": "info", "event": "Loading embedding model"}
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4904.56it/s]
{"documents": 3, "path": "faiss_index\\session_20260724_012123_abdce555\\faiss_index", "timestamp": "2026-07-23T19:51:30.474667Z", "level": "info", "event": "Created FAISS index"}
{"count": 3, "timestamp": "2026-07-23T19:51:30.555663Z", "level": "info", "event": "Added documents to FAISS"}
{"search_type": "mmr", "timestamp": "2026-07-23T19:51:30.557668Z", "level": "info", "event": "Retriever created"}
{"provider": "groq", "model": "llama-3.3-70b-versatile", "timestamp": "2026-07-23T19:51:30.560662Z", "level": "info

Q2: What are the key components of an Agentic AI system?
A2: The key components of an Agentic AI system include:

1. Reasoning: deciding what to do next.
2. Memory: retaining useful context across steps.
3. Tools: calling APIs, calculators, databases, OCR systems, or web search.
4. Planning: decomposing large objectives into smaller tasks.
5. Reflection: evaluating outputs and correcting mistakes.

--------------------------------------------------------------------------------



{"count": 1, "failed": 0, "timestamp": "2026-07-23T19:51:40.127333Z", "level": "info", "event": "Documents loaded"}
{"documents": 1, "chunks": 3, "timestamp": "2026-07-23T19:51:40.140332Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-07-23T19:51:40.141333Z", "level": "info", "event": "Loading embedding model"}
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3961.24it/s]
{"documents": 3, "path": "faiss_index\\session_20260724_012139_d6b5286b\\faiss_index", "timestamp": "2026-07-23T19:51:46.176681Z", "level": "info", "event": "Created FAISS index"}
{"count": 3, "timestamp": "2026-07-23T19:51:46.259678Z", "level": "info", "event": "Added documents to FAISS"}
{"search_type": "mmr", "timestamp": "2026-07-23T19:51:46.260681Z", "level": "info", "event": "Retriever created"}
{"provider": "groq", "model": "llama-3.3-70b-versatile", "timestamp": "2026-07-23T19:51:46.263686Z", "level": "info

Q3: What are the common applications and challenges of Agentic AI?
A3: The common applications of Agentic AI include: 
1. Customer support
2. Document assistants
3. Research automation
4. Coding assistants
5. Workflow automation
6. Enterprise knowledge management

The challenges of Agentic AI include: 
1. Hallucinations
2. Tool failures
3. Security risks
4. Latency
5. Cost

--------------------------------------------------------------------------------



built in evl function using langsmith

In [39]:
from langsmith.evaluation import evaluate

# Define a custom evaluator function
def qa_evaluator(run, example):
    """
    Custom QA evaluator that checks if the answer matches the expected answer.
    """
    # Extract prediction and reference
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    
    # Simple scoring - you can make this more sophisticated
    if prediction.lower() == reference.lower():
        score = 1.0
    else:
        score = 0.5  # Partial credit
    
    return {
        "key": "qa_correctness",
        "score": score,
        "comment": f"Prediction: {prediction[:100]}..."
    }

dataset_name = "Rag_App_dataset"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[qa_evaluator],
    experiment_prefix="test-agenticAIReport-qa-rag",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-305227ab' at:
https://smith.langchain.com/o/c035d350-7f74-44a1-b0e2-c8a49bce5409/datasets/8f330df3-08b1-45c8-9e7a-688dbc6e3bce/compare?selectedSessions=f109c3c0-a93f-4429-ac63-3cc80e65294d




3it [00:00,  6.59it/s]


custm eval funcntion

In [62]:
from langsmith.evaluation import evaluate
from langsmith.schemas import Run, Example
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

load_dotenv()


def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Returns:
        dict with 'key' and 'score' (1 for correct, 0 for incorrect)
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM - Using Groq
    try:
        llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0,
            api_key=os.getenv("GROQ_API_KEY")
        )
    except Exception:
        # Fallback to Google if Groq fails
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash-exp",
            temperature=0,
            google_api_key=os.getenv("GOOGLE_API_KEY")
        )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        verdict = ""
        for line in response_text.split('\n'):
            if line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
                break
        
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0
        }


# ============================================================
# RUN EVALUATION
# ============================================================

experiment_results = evaluate(
    answer_ai_report_question,  # Your RAG function
    data="Rag_App_dataset",
    evaluators=[correctness_evaluator],
    experiment_prefix="agenticAIReport-correctness-eval",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "llama-3.3-70b-versatile",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\n✅ Evaluation completed!")
print(f"📊 Results: {experiment_results}")
print("🔗 Check the LangSmith UI for detailed results.")

View the evaluation results for experiment: 'agenticAIReport-correctness-eval-00499588' at:
https://smith.langchain.com/o/c035d350-7f74-44a1-b0e2-c8a49bce5409/datasets/8f330df3-08b1-45c8-9e7a-688dbc6e3bce/compare?selectedSessions=d67a2180-5470-46e2-be7f-7e6edeb14c09




3it [00:04,  1.59s/it]


✅ Evaluation completed!
📊 Results: <ExperimentResults agenticAIReport-correctness-eval-00499588>
🔗 Check the LangSmith UI for detailed results.
